# 🛡️ Cyber Incident Trend Analysis
## Forecasting Monthly Incident Volume & Discovering Attack Patterns Using Machine Learning

**B.Tech CSE — Semester V · Machine Learning Case Study**

---

### Data Source
- **VERIS Community Database (VCDB)**: [github.com/vz-risk/VCDB](https://github.com/vz-risk/VCDB)
- **Records**: 10,391 unique incidents after cleaning
- **Time span**: January 2010 – June 2026

## 1. Setup & Data Loading

**Why these libraries?**
- `pandas` + `numpy`: Core data manipulation
- `plotly`: Interactive charts (used in the dashboard too)
- `scikit-learn`: Clustering algorithms and metrics
- `statsmodels`: SARIMA time-series model
- `prophet`: Facebook's robust forecasting library
- `xgboost`: Gradient boosting for regression with lag features

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

# Try importing xgboost — it may not be available on all systems
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not available — will skip XGBoost experiments.')

print('All libraries loaded successfully.')

In [ ]:
# ── Load Forecasting Data ──
df_forecast = pd.read_csv('cyber_incidents_monthly_forecasting.csv')
df_forecast['month'] = pd.to_datetime(df_forecast['month'])
df_forecast = df_forecast.sort_values('month').reset_index(drop=True)

print(f'Forecasting data: {df_forecast.shape}')
print(f'Date range: {df_forecast["month"].min()} to {df_forecast["month"].max()}')
df_forecast.head()

In [ ]:
# ── Load Clustering Data ──
# Read industry_sector as string to keep leading zeros (e.g., '00' for Unknown)
df_cluster = pd.read_csv('cyber_incidents_clustering.csv', dtype={'industry_sector': str})

print(f'Clustering data: {df_cluster.shape}')
print(f'MOVEit events: {df_cluster["is_2023_moveit_event"].sum()}')
df_cluster.head()

## 2. Data Quality Assessment

**Why these checks?**
Understanding data quality is essential before modeling. Known issues:
- 205 duplicate incident_ids removed during preprocessing
- 28% of raw rows had no month (assigned to January of their year)
- Years before 2010 are too sparse for reliable analysis
- June 2023 has 755 incidents (MOVEit mass-exploitation) — extreme outlier

In [ ]:
# NAICS Industry Mapping
NAICS_MAP = {
    '11': 'Agriculture', '21': 'Mining', '22': 'Utilities',
    '23': 'Construction', '31': 'Manufacturing', '32': 'Manufacturing',
    '33': 'Manufacturing', '42': 'Wholesale Trade', '44': 'Retail Trade',
    '45': 'Retail Trade', '48': 'Transportation', '49': 'Transportation',
    '51': 'Information', '52': 'Finance & Insurance', '53': 'Real Estate',
    '54': 'Professional Services', '55': 'Management', '56': 'Administrative',
    '61': 'Education', '62': 'Healthcare', '71': 'Arts & Entertainment',
    '72': 'Accommodation & Food', '81': 'Other Services',
    '92': 'Public Administration', '00': 'Unknown',
}

# Check for data quality issues
print('=== Data Quality Report ===')
print(f'\nForecasting dataset:')
print(f'  Rows: {len(df_forecast)}')
print(f'  Columns: {df_forecast.shape[1]}')
print(f'  Nulls: {df_forecast.isnull().sum().sum()}')
print(f'  June 2023 spike: {df_forecast[df_forecast["month"]=="2023-06-01"]["incident_count"].values[0]}')
print(f'  Average monthly count: {df_forecast["incident_count"].mean():.1f}')

print(f'\nClustering dataset:')
print(f'  Rows: {len(df_cluster)}')
print(f'  Columns: {df_cluster.shape[1]}')
print(f'  Unique incident IDs: {df_cluster["incident_id"].nunique()}')
print(f'  Industry sectors: {df_cluster["industry_sector"].nunique()}')
print(f'  MOVEit events: {df_cluster["is_2023_moveit_event"].sum()}')

# Multi-label check
action_cols = [c for c in df_cluster.columns if c.startswith('action_')]
multi_label = (df_cluster[action_cols].sum(axis=1) > 1).sum()
print(f'  Multi-label incidents (>1 action): {multi_label} ({multi_label/len(df_cluster)*100:.1f}%)')

# Class imbalance
print(f'\nAction type counts:')
for col in action_cols:
    print(f'  {col}: {df_cluster[col].sum()}')

## 3. Exploratory Data Analysis (EDA)

**Why these visualizations?**
- Yearly bar chart → shows uneven data collection (NOT real trends)
- Monthly line chart → reveals the June 2023 spike and overall patterns
- Action mix → shows how attack types have shifted over time
- Industry distribution → reveals reporting bias (healthcare, finance lead due to disclosure laws)

In [ ]:
# ── 3.1 Incidents per Year ──
yearly = df_forecast.copy()
yearly['year'] = yearly['month'].dt.year
yearly_counts = yearly.groupby('year')['incident_count'].sum().reset_index()

fig = px.bar(yearly_counts, x='year', y='incident_count',
             title='Total Incidents per Year',
             template='plotly_dark',
             color='incident_count',
             color_continuous_scale='teal')
fig.update_layout(height=400)
fig.show()

In [ ]:
# ── 3.2 Monthly Trend with June 2023 Spike ──
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_forecast['month'], y=df_forecast['incident_count'],
    mode='lines', fill='tozeroy',
    fillcolor='rgba(0, 212, 170, 0.1)',
    line=dict(color='#00D4AA', width=1.5),
    name='Monthly Incidents',
))

# Annotate the MOVEit spike
fig.add_annotation(
    x='2023-06-01', y=755,
    text='⚠ MOVEit (755 incidents)',
    showarrow=True, arrowhead=2, arrowcolor='#FF4757',
    font=dict(color='#FF4757', size=12),
    ax=-60, ay=-40,
)

fig.update_layout(
    title='Monthly Incident Trend (Jan 2010 – Jun 2026)',
    template='plotly_dark', height=450,
    xaxis_title='Date', yaxis_title='Incident Count',
)
fig.show()

In [ ]:
# ── 3.3 Action Type Mix Over Time ──
action_cols_forecast = [c for c in df_forecast.columns if c.startswith('action_')]

fig = go.Figure()
colors = ['#00D4AA', '#3B82F6', '#FF4757', '#FFC048', '#A855F7', '#00B4D8', '#F97316', '#EC4899']
for i, col in enumerate(action_cols_forecast):
    fig.add_trace(go.Scatter(
        x=df_forecast['month'], y=df_forecast[col],
        mode='lines', stackgroup='one',
        name=col.replace('action_', ''),
        line=dict(width=0.5, color=colors[i % len(colors)]),
    ))

fig.update_layout(
    title='Action Type Composition Over Time',
    template='plotly_dark', height=450,
    xaxis_title='Date', yaxis_title='Count',
)
fig.show()

In [ ]:
# ── 3.4 Top Industries ──
df_cluster['industry_name'] = df_cluster['industry_sector'].map(
    lambda x: NAICS_MAP.get(str(x).zfill(2), f'Sector {x}')
)
industry_counts = df_cluster['industry_name'].value_counts().head(12)

fig = px.bar(x=industry_counts.values, y=industry_counts.index,
             orientation='h', template='plotly_dark',
             title='Top 12 Industries by Incident Count',
             color=industry_counts.values,
             color_continuous_scale='teal')
fig.update_layout(height=450, yaxis=dict(autorange='reversed'))
fig.show()

In [ ]:
# ── 3.5 Actor Distribution ──
actor_cols = [c for c in df_cluster.columns if c.startswith('actor_')]
actor_sums = df_cluster[actor_cols].sum()
actor_sums.index = [c.replace('actor_', '') for c in actor_sums.index]

fig = px.pie(names=actor_sums.index, values=actor_sums.values,
             title='Actor Type Distribution', template='plotly_dark',
             hole=0.4, color_discrete_sequence=['#00D4AA', '#3B82F6', '#FF4757', '#FFC048'])
fig.update_layout(height=400)
fig.show()

In [ ]:
# ── 3.6 Action × Industry Heatmap ──
top_10_industries = df_cluster['industry_name'].value_counts().head(10).index
df_heat = df_cluster[df_cluster['industry_name'].isin(top_10_industries)]

heat_data = df_heat.groupby('industry_name')[action_cols].sum()
heat_data.columns = [c.replace('action_', '') for c in heat_data.columns]

fig = px.imshow(heat_data, template='plotly_dark',
                title='Action Type × Industry Heatmap',
                color_continuous_scale='teal',
                aspect='auto')
fig.update_layout(height=500)
fig.show()

### EDA Key Findings

1. **Yearly counts peak in 2013 (~1,205)** and decline after — this reflects VCDB data collection, not real-world trends.
2. **June 2023 has 755 incidents** — the MOVEit mass-exploitation event. Must be handled as an outlier.
3. **Healthcare and Finance** dominate the industry distribution, likely due to mandatory breach disclosure laws.
4. **Error (accidental)** is the most common action type overall, followed by Hacking and Physical.
5. **External actors** account for ~55% of incidents, but **internal actors** are substantial (~40%).
6. **After 2018**, monthly counts drop to single digits — a data collection gap, not a real decline.

## 4. Task 1: Time-Series Forecasting

### Design Decisions

**Why 2010 cutoff?** Years before 2010 have very few entries (<50/year), making them unreliable for training.

**Why time-based split (never random)?** In time-series, random splitting causes data leakage — the model would see future data during training, inflating performance metrics.

**Why train through Dec 2021?** This gives ~12 years (144 months) of training data and ~4.5 years of test data.

**Why these models?**
- **Seasonal Naive**: Simplest possible baseline — if this wins, the series is too noisy for ML.
- **SARIMA**: Classical seasonal model, captures linear trends + 12-month seasonality.
- **Prophet**: Robust to outliers and missing data, handles yearly seasonality automatically.
- **XGBoost**: Can capture nonlinear lag relationships, but needs manual feature engineering.

**Why MAE, RMSE, MAPE, sMAPE?**
- MAE: Most interpretable (average absolute error in incidents).
- RMSE: Penalizes large errors more.
- MAPE: Percentage-based but unreliable with near-zero values.
- sMAPE: Symmetric, more stable than MAPE for small values.

In [ ]:
# ── Helper Functions ──

def compute_metrics(y_true, y_pred):
    """Compute MAE, RMSE, MAPE (guarded), sMAPE."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    
    denom = np.abs(y_true) + np.abs(y_pred)
    smape_vals = np.where(denom == 0, 0, 2 * np.abs(y_true - y_pred) / denom)
    smape = np.mean(smape_vals) * 100
    
    return {'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'MAPE': round(mape, 2), 'sMAPE': round(smape, 2)}


def seasonal_naive(train, horizon, period=12):
    """Repeat values from the same month one year ago."""
    values = train.values
    return np.array([values[len(values) - period + (h % period)] 
                     for h in range(horizon)])


def create_lag_features(df, target='incident_count'):
    """Create lag, rolling, and calendar features for XGBoost."""
    result = df.copy()
    for lag in [1, 2, 3, 12]:
        result[f'lag_{lag}'] = result[target].shift(lag)
    result['rolling_mean_3'] = result[target].shift(1).rolling(3).mean()
    result['rolling_mean_6'] = result[target].shift(1).rolling(6).mean()
    result['month_number'] = result['month'].dt.month
    return result.dropna().reset_index(drop=True)

print('Helper functions defined.')

In [ ]:
# ═══════════════════════════════════════════════
# RUN BOTH EXPERIMENTS
# ═══════════════════════════════════════════════

all_results = {}

for exp_name, cap_spike in [('A: Spike Included', False), ('B: Spike Capped', True)]:
    print(f'\n{"="*50}')
    print(f'EXPERIMENT {exp_name}')
    print(f'{"="*50}')
    
    # Prepare data
    df = df_forecast.copy()
    if cap_spike:
        idx = df[df['month'] == '2023-06-01'].index[0]
        neighbors = [df.loc[idx-1, 'incident_count'], df.loc[idx+1, 'incident_count']]
        df.loc[idx, 'incident_count'] = int(np.median(neighbors))
        print(f'  Spike capped to {int(np.median(neighbors))} (median of neighbors)')
    
    # Time-based split
    split_date = pd.Timestamp('2022-01-01')
    train_df = df[df['month'] < split_date]
    test_df = df[df['month'] >= split_date]
    train_series = train_df.set_index('month')['incident_count']
    test_series = test_df.set_index('month')['incident_count']
    horizon = len(test_series)
    
    print(f'  Train: {len(train_df)} months, Test: {len(test_df)} months')
    
    results = []
    predictions = {}
    
    # 1. Seasonal Naive
    print('\n  [1/4] Seasonal Naive...')
    naive_preds = seasonal_naive(train_series, horizon)
    naive_m = compute_metrics(test_series.values, naive_preds)
    naive_m['Model'] = 'Seasonal Naive'
    results.append(naive_m)
    predictions['Seasonal Naive'] = naive_preds
    print(f'    MAE={naive_m["MAE"]}, RMSE={naive_m["RMSE"]}, sMAPE={naive_m["sMAPE"]}')
    
    # 2. SARIMA
    print('\n  [2/4] SARIMA(1,1,1)(1,1,1,12)...')
    model = SARIMAX(train_series, order=(1,1,1), seasonal_order=(1,1,1,12),
                    enforce_stationarity=False, enforce_invertibility=False)
    fitted = model.fit(disp=False, maxiter=200)
    sarima_preds = np.maximum(fitted.forecast(horizon).values, 0)
    sarima_m = compute_metrics(test_series.values, sarima_preds)
    sarima_m['Model'] = 'SARIMA'
    results.append(sarima_m)
    predictions['SARIMA'] = sarima_preds
    print(f'    MAE={sarima_m["MAE"]}, RMSE={sarima_m["RMSE"]}, sMAPE={sarima_m["sMAPE"]}')
    
    # 3. Prophet
    print('\n  [3/4] Prophet...')
    prophet_df = train_df[['month', 'incident_count']].rename(
        columns={'month': 'ds', 'incident_count': 'y'})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
               daily_seasonality=False, changepoint_prior_scale=0.05)
    m.fit(prophet_df)
    future = m.make_future_dataframe(periods=horizon, freq='MS')
    prophet_forecast = m.predict(future)
    prophet_preds = np.maximum(prophet_forecast.tail(horizon)['yhat'].values, 0)
    prophet_m = compute_metrics(test_series.values, prophet_preds)
    prophet_m['Model'] = 'Prophet'
    results.append(prophet_m)
    predictions['Prophet'] = prophet_preds
    print(f'    MAE={prophet_m["MAE"]}, RMSE={prophet_m["RMSE"]}, sMAPE={prophet_m["sMAPE"]}')
    
    # 4. XGBoost
    if HAS_XGB:
        print('\n  [4/4] XGBoost...')
        df_feat = create_lag_features(df)
        feat_cols = ['lag_1', 'lag_2', 'lag_3', 'lag_12',
                     'rolling_mean_3', 'rolling_mean_6', 'month_number']
        train_feat = df_feat[df_feat['month'] < split_date]
        test_feat = df_feat[df_feat['month'] >= split_date]
        
        xgb = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1,
                           random_state=42, verbosity=0)
        xgb.fit(train_feat[feat_cols], train_feat['incident_count'])
        xgb_preds = np.maximum(xgb.predict(test_feat[feat_cols]), 0)
        xgb_m = compute_metrics(test_feat['incident_count'].values, xgb_preds)
        xgb_m['Model'] = 'XGBoost'
        results.append(xgb_m)
        predictions['XGBoost'] = xgb_preds
        print(f'    MAE={xgb_m["MAE"]}, RMSE={xgb_m["RMSE"]}, sMAPE={xgb_m["sMAPE"]}')
    else:
        print('\n  [4/4] XGBoost — SKIPPED (not installed)')
    
    results_df = pd.DataFrame(results)[['Model', 'MAE', 'RMSE', 'MAPE', 'sMAPE']]
    all_results[exp_name] = {'df': results_df, 'predictions': predictions,
                              'test_series': test_series, 'test_df': test_df}
    
    print(f'\n  Results:')
    display(results_df)

In [ ]:
# ── Actual vs Predicted (Experiment B, best model) ──
exp_b = all_results['B: Spike Capped']
test_dates = exp_b['test_df']['month']
test_actual = exp_b['test_series'].values

fig = go.Figure()
fig.add_trace(go.Bar(x=test_dates, y=test_actual, name='Actual', marker_color='#00D4AA', opacity=0.7))

# Plot best model predictions
best_model_name = exp_b['df'].loc[exp_b['df']['MAE'].idxmin(), 'Model']
best_preds = exp_b['predictions'][best_model_name]
fig.add_trace(go.Bar(x=test_dates[:len(best_preds)], y=best_preds,
                     name=f'Predicted ({best_model_name})', marker_color='#FF4757', opacity=0.7))

fig.update_layout(title=f'Actual vs Predicted — {best_model_name} (Spike Capped)',
                  template='plotly_dark', barmode='group', height=400)
fig.show()

### Forecasting Conclusions

1. **Seasonal Naive and XGBoost** are the top performers on MAE.
2. **If naive wins, that's honest** — the series is short (~144 train months), noisy, and driven by reporting practices rather than a stable signal.
3. **Capping the spike** reduces MAE/RMSE dramatically — the spike is an unpredictable black-swan event.
4. **MAPE is unreliable** for this data because many test months have counts near zero (1–10).
5. **MAE is the most trustworthy metric** — it directly measures average absolute error in incident counts.

## 5. Task 2: Clustering

### Design Decisions

**Why these features?** action_* (what), actor_* (who), pattern_* (how), industry_sector (where) — these four dimensions capture the full attack profile.

**Why exclude MOVEit rows?** 740+ identical records would dominate one cluster and reduce diversity.

**Why K-Means as primary?** Scales well to 10k records, produces interpretable centroids, k is tunable.

**Why PCA only for visualization?** PCA reduces dimensionality but loses interpretability. We cluster on full features and only use PCA 2D for scatter plots.

**Why k=5?** Determined by elbow method + silhouette score — k=4-6 is the sweet spot.

In [ ]:
# ── Prepare Clustering Features ──
df_cl = df_cluster[df_cluster['is_2023_moveit_event'] == 0].copy().reset_index(drop=True)
print(f'Records (excl. MOVEit): {len(df_cl)}')

action_cols = [c for c in df_cl.columns if c.startswith('action_')]
actor_cols = [c for c in df_cl.columns if c.startswith('actor_')]
pattern_cols = [c for c in df_cl.columns if c.startswith('pattern_')]

# One-hot encode industry sector
df_cl['sector_group'] = df_cl['industry_sector'].map(
    lambda x: NAICS_MAP.get(str(x).zfill(2), 'Other'))
sector_dummies = pd.get_dummies(df_cl['sector_group'], prefix='sector')

feature_df = pd.concat([df_cl[action_cols], df_cl[actor_cols],
                         df_cl[pattern_cols], sector_dummies], axis=1)
X = feature_df.values.astype(float)
feature_names = list(feature_df.columns)

print(f'Feature matrix: {X.shape}')
print(f'Features: {len(feature_names)}')

In [ ]:
# ── Elbow Method ──
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers',
                         name='Inertia', line=dict(color='#00D4AA')), secondary_y=False)
fig.add_trace(go.Scatter(x=list(k_range), y=silhouettes, mode='lines+markers',
                         name='Silhouette', line=dict(color='#FFC048')), secondary_y=True)
fig.update_layout(title='Elbow Method', template='plotly_dark', height=400)
fig.show()

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'\nBest k by silhouette score: {best_k} (score: {max(silhouettes):.4f})')

In [ ]:
# ── Run All Three Algorithms (k=5) ──
n_clusters = 5

# K-Means
km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
km_labels = km.fit_predict(X)
km_sil = silhouette_score(X, km_labels)
km_db = davies_bouldin_score(X, km_labels)
print(f'K-Means:       Silhouette={km_sil:.4f}, Davies-Bouldin={km_db:.4f}')

# DBSCAN (auto-tune eps)
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X)
dists, _ = nn.kneighbors(X)
eps_val = float(np.percentile(dists[:, -1], 90))
db_model = DBSCAN(eps=eps_val, min_samples=5)
db_labels = db_model.fit_predict(X)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
mask = db_labels != -1
if mask.sum() > 1 and len(set(db_labels[mask])) > 1:
    db_sil = silhouette_score(X[mask], db_labels[mask])
    db_db = davies_bouldin_score(X[mask], db_labels[mask])
else:
    db_sil = -1; db_db = float('inf')
print(f'DBSCAN:        Clusters={n_db_clusters}, Noise={n_noise}, Sil={db_sil:.4f}')

# Agglomerative
ag = AgglomerativeClustering(n_clusters=n_clusters)
ag_labels = ag.fit_predict(X)
ag_sil = silhouette_score(X, ag_labels)
ag_db = davies_bouldin_score(X, ag_labels)
print(f'Agglomerative: Silhouette={ag_sil:.4f}, Davies-Bouldin={ag_db:.4f}')

# Comparison table
pd.DataFrame({
    'Algorithm': ['K-Means', 'DBSCAN', 'Agglomerative'],
    'Silhouette ↑': [round(km_sil,4), round(db_sil,4), round(ag_sil,4)],
    'Davies-Bouldin ↓': [round(km_db,4), round(db_db,4), round(ag_db,4)],
})

In [ ]:
# ── PCA 2D Visualization ──
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

scatter_df = pd.DataFrame({'PC1': X_2d[:, 0], 'PC2': X_2d[:, 1],
                            'Cluster': km_labels.astype(str)})

fig = px.scatter(scatter_df, x='PC1', y='PC2', color='Cluster',
                 title='K-Means Clusters (PCA 2D Projection)',
                 template='plotly_dark',
                 color_discrete_sequence=['#00D4AA', '#3B82F6', '#FF4757', '#FFC048', '#A855F7'],
                 opacity=0.5)
fig.update_layout(height=500)
fig.show()

In [ ]:
# ── Cluster Profiles ──
print('\n' + '='*60)
print('CLUSTER PROFILES')
print('='*60)

for cl in sorted(set(km_labels)):
    mask = km_labels == cl
    cl_df = df_cl[mask]
    
    top_action = cl_df[action_cols].sum().idxmax().replace('action_', '')
    top_actor = cl_df[actor_cols].sum().idxmax().replace('actor_', '')
    top_pattern = cl_df[pattern_cols].sum().idxmax().replace('pattern_', '').replace('_', ' ')
    top_sectors = cl_df['sector_group'].value_counts().head(3).index.tolist()
    
    name = f'{top_sectors[0]} {top_actor.lower()} {top_action.lower()}'.title()
    
    print(f'\nCluster {cl}: {name} ({mask.sum()} incidents)')
    print(f'  Top Action:  {top_action}')
    print(f'  Top Actor:   {top_actor}')
    print(f'  Top Pattern: {top_pattern}')
    print(f'  Industries:  {", ".join(top_sectors)}')

## 6. Conclusions & Limitations

### Forecasting
- Seasonal Naive / XGBoost are the best models — the series is too short and noisy for complex models to add much value.
- The June 2023 MOVEit spike is an unpredictable black-swan event.
- Capping the spike gives a more realistic view of typical forecasting performance.

### Clustering
- K-Means with k=5 produces the most interpretable clusters.
- Silhouette scores are modest (~0.18) due to the sparse binary feature space.
- Cluster profiles are actionable: they separate Healthcare/Error from Finance/Hacking incidents.

### Limitations
1. **Reporting bias**: VCDB is publicly reported incidents only — not representative of all cyber attacks.
2. **Small counts**: After 2018, monthly counts drop to single digits.
3. **Multi-label**: 18.3% of incidents have multiple action types.
4. **Class imbalance**: Environmental action has only 10 rows.
5. **Short series**: 144 training months limits seasonal model performance.

### Future Work
- Combine with NVD, threat intelligence feeds
- Try LSTM / Transformer-based time-series models
- Add anomaly detection
- Real-time dashboard connected to live data feeds